# AuctionIQ — Recommender Analysis

Academic notebook for dataset exploration, feature engineering, similarity, personalized recommendation, diversity and evaluation.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))
from src.data_loader import load_data
from src.preprocessing import clean_data, dataset_summary
from src.feature_engineering import build_player_features, add_recency_scores
from src.player_profiles import build_profile_scores
from src.hybrid_recommender import AuctionIQRecommender


In [ ]:
deliveries, matches = load_data(ROOT/"data")
deliveries, matches = clean_data(deliveries, matches)
dataset_summary(deliveries, matches)


## 1. Data exploration

In [ ]:
print("Deliveries:", deliveries.shape)
print("Matches:", matches.shape)
print("Seasons:", matches["season"].nunique())
print("Players:", len(set(deliveries["batter"].dropna()) | set(deliveries["bowler"].dropna()) | set(deliveries["non_striker"].dropna())))
matches["season"].value_counts().sort_index()


## 2. Feature engineering

In [ ]:
features = build_player_features(deliveries, matches)
features = add_recency_scores(features, deliveries, matches, "Last 2 seasons")
profiles = build_profile_scores(features)
profiles.head()


## 3. Content-based model and cosine similarity

In [ ]:
recommender = AuctionIQRecommender(profiles)
recommender.content_features


In [ ]:
player = profiles.sort_values("runs", ascending=False).iloc[0]["player"]
print("Reference player:", player)
recommender.similar(player, 5)


## 4. Personalized recommendation

In [ ]:
recs = recommender.recommend(
    required_role="All-Rounder",
    batting_importance=0.8,
    bowling_importance=0.7,
    recent_importance=0.9,
    consistency_importance=0.7,
    risk="Medium",
    top_k=5
)
recs[["rank","player","role","final_score","confidence"]]


## 5. Explain a recommendation

In [ ]:
from src.explainability import explain_recommendation
row = recs.iloc[0]
print(row["player"], row["final_score"])
for reason in explain_recommendation(row):
    print("✓", reason)


## 6. Diversity

In [ ]:
from src.evaluation import intra_list_diversity
feature_cols = ["batting_strength","bowling_strength","recent_form_score","consistency_score","powerplay_score","middle_score","death_score"]
intra_list_diversity(recs, feature_cols)


## 7. Limitations

There is no actual auction price, nationality, official role, user rating, or auction outcome field. Evaluation must therefore be framed as proxy/offline evaluation rather than auction-prediction accuracy.